In [2]:
import requests

url = "https://github.com/sue-ellenn/OvP/releases/download/datav1/search.db"

r = requests.get(url)
r.raise_for_status()

print("Grootte:", len(r.content) / 1024**2, "MB")

Grootte: 253.11328125 MB


In [2]:
import pandas as pd

df = pd.read_csv("db_csv/employee.csv")
df.head(10)


,id,name,url,faculties,keywords,onderzoeksthema,onderzoeksgroep,publicaties,onderwijs
0,1,M. a.b maar ma,https://www.ru.nl/personen/a-b-m,oudpromovendus economisch sociaal en demografi...,trefwoord hanze,NaN,NaN,2018 -\n\n 2023\n communicatie tussen hanzeste...,NaN
1,2,H. van der aa dirkje ma,https://www.ru.nl/personen/aa-h-van-der,projectleider radboud docent academie,NaN,NaN,NaN,NaN,NaN
2,3,p.m. van der aa ilse,https://www.ru.nl/personen/aa-p-van-der,hoofd hr salarisadministratie,NaN,NaN,NaN,NaN,NaN
3,4,b.j.g.b aalber Bob,https://www.ru.nl/personen/aalbers-b,medewerker food beverage support,NaN,NaN,NaN,NaN,NaN
4,5,drs. p.a.t aalber Remco,https://www.ru.nl/personen/aalbers-p,medewerker fnwi computer- en communicatiezaken,trefwoord internet open source software,NaN,NaN,NaN,NaN
5,6,prof. dr. H.l. aalber Rick,https://www.ru.nl/personen/aalbers-h,hoogleraar strategie,none trefwoord reorganisatie downsizing intra-...,NaN,NaN,Ellis J. khanaghum s. aalber R. Tuertscher p. ...,2012 -bko and uko qualified basic and advanced...
6,7,j.m Aaldering Jordy,https://www.ru.nl/personen/aaldering-j,promovendus software science,NaN,NaN,NaN,NaN,advanced programming extern link https://ru.os...
7,8,D. van Aalst Dick,https://www.ru.nl/personen/aalst-d-van,beeldredacteur marketing communications,NaN,NaN,NaN,NaN,NaN
8,9,t. van Aanholt ted ma,https://www.ru.nl/personen/aanholt-t-van,promovendus geschiedenis van de filosofie,trefwoord wetenschapsgeschiedenis geschiedenis...,NaN,center for the History of philosophy and scien...,NaN,geschiedenis van de filosofie 2(externe link h...
9,10,L. van aard-krommenhoek linsey,https://www.ru.nl/personen/aard-krommenhoek-l-van,hr medewerker hr flexpool,NaN,NaN,NaN,NaN,NaN


In [4]:

s = df["keywords"].str.contains('neuromorphic').sum()
print(s)

3


In [5]:
import sqlite3

conn = sqlite3.connect("data/search.db")

term = "neuromorphic"

print(conn.execute(
    "SELECT name, keywords FROM employee WHERE keywords LIKE ?",
    (f"%{term}%",)
).fetchall())

[('prof. dr. m.a.j Van gerven Marcel', 'none trefwoord neural Networks machine learning neuromorphic computing'), ('M. karamimanesh mehrzad phd', 'trefwoord fpga neuromorphic computing spiking neural Networks'), ('dr. j.h Mentink Johan', 'trefwoord natuurkunde magnetisme quantum veel-deeltjes effect machine learning neuromorphic computing')]


In [7]:
# import sqlite3

conn = sqlite3.connect("data/search.db")

tables = conn.execute("""
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
    ORDER BY name
""").fetchall()

for table in tables:
    print(table)

conn.close()

('course_employee', 'table')
('employee', 'table')
('employee_index', 'table')
('osiris', 'table')
('repo', 'table')
('search', 'table')
('search_config', 'table')
('search_content', 'table')
('search_data', 'table')
('search_docsize', 'table')
('search_idx', 'table')


In [10]:
conn = sqlite3.connect("data/search.db")

print(conn.execute("""
    SELECT sql
    FROM sqlite_master
    WHERE name = 'search'
""").fetchone()[0])

# conn.close()

CREATE VIRTUAL TABLE search
USING fts5(name, source, text)


In [11]:
print(conn.execute("PRAGMA table_info(employee)").fetchall())

[(0, 'id', 'INTEGER', 0, None, 1), (1, 'name', 'TEXT', 0, None, 0), (2, 'url', 'TEXT', 0, None, 0), (3, 'faculties', 'TEXT', 0, None, 0), (4, 'keywords', 'TEXT', 0, None, 0), (5, 'onderzoeksthema', 'TEXT', 0, None, 0), (6, 'onderzoeksgroep', 'TEXT', 0, None, 0), (7, 'publicaties', 'TEXT', 0, None, 0), (8, 'onderwijs', 'TEXT', 0, None, 0)]


In [13]:
import sqlite3

conn = sqlite3.connect("data/search.db")

term = "neuromorphic"

print("EMPLOYEE:")
print(conn.execute(
    "SELECT name, keywords FROM employee WHERE keywords LIKE ?",
    (f"%{term}%",)
).fetchall())

print("\nSEARCH:")
print(conn.execute(
    "SELECT name, source, text FROM search WHERE search MATCH ?",
    (term,)
).fetchall())

conn.close()

EMPLOYEE:
[('prof. dr. m.a.j Van gerven Marcel', 'none trefwoord neural Networks machine learning neuromorphic computing'), ('M. karamimanesh mehrzad phd', 'trefwoord fpga neuromorphic computing spiking neural Networks'), ('dr. j.h Mentink Johan', 'trefwoord natuurkunde magnetisme quantum veel-deeltjes effect machine learning neuromorphic computing')]

SEARCH:
[('nwi-nm085c', 'Osiris', 'after successful completion of the course you will be able to understand the computational neuroscience literature in terms of being able to implement and critically evaluate these model in terms of whether they address the underlying neuroscience problem and write computer programs to implement a number of common computational models after successful completion of the course you will be able to understand the computational neuroscience literature in terms of being able to implement and critically evaluate these model in terms of whether they address the underlying neuroscience problem and write compute